In [ ]:
!pip install -q transformers datasets evaluate seqeval scikit-learn pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.3 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import random
import numpy as np
import pandas as pd

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from collections import Counter

input_txt_path = "/content/oe_ner_dataset_cleaned.txt"

def inspect_dataset(file_path):
    sentence_count = 0
    token_count = 0

    label_counter = Counter()
    unique_labels = set()

    invalid_lines = []
    sentence_lengths = []

    current_tokens = []
    current_labels = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.rstrip("\n")

            # sentence boundary
            if not line.strip():
                if current_tokens:
                    sentence_count += 1
                    sentence_lengths.append(len(current_tokens))
                    current_tokens = []
                    current_labels = []
                continue

            parts = line.split()

            if len(parts) < 2:
                invalid_lines.append((line_num, line))
                continue

            token = " ".join(parts[:-1])
            label = parts[-1]

            current_tokens.append(token)
            current_labels.append(label)

            token_count += 1
            label_counter[label] += 1
            unique_labels.add(label)

    # last sentence
    if current_tokens:
        sentence_count += 1
        sentence_lengths.append(len(current_tokens))

    print("===== DATASET SUMMARY =====")
    print(f"Total sentences: {sentence_count}")
    print(f"Total tokens: {token_count}")

    print("\nLabels found:")
    print(unique_labels)

    print("\nLabel distribution:")
    print(dict(label_counter))

    if sentence_lengths:
        print("\nSentence length stats:")
        print(f"Min: {min(sentence_lengths)}")
        print(f"Max: {max(sentence_lengths)}")
        print(f"Avg: {sum(sentence_lengths)/len(sentence_lengths):.2f}")

    print(f"\nInvalid lines: {len(invalid_lines)}")

    if invalid_lines[:10]:
        print("\nSample invalid lines:")
        for item in invalid_lines[:10]:
            print(item)

inspect_dataset(input_txt_path)

===== DATASET SUMMARY =====
Total sentences: 900
Total tokens: 19695

Labels found:
{'B-PER', 'O'}

Label distribution:
{'O': 18457, 'B-PER': 1238}

Sentence length stats:
Min: 4
Max: 86
Avg: 21.88

Invalid lines: 0


In [ ]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

input_txt_path = "/content/oe_ner_dataset_cleaned.txt"
seed = 42

def read_word_tag_dataset(file_path):
    examples = []
    tokens = []
    labels = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            # sentence boundary
            if not line.strip():
                if tokens:
                    examples.append({
                        "tokens": tokens,
                        "labels": labels
                    })
                    tokens = []
                    labels = []
                continue

            parts = line.split()
            if len(parts) < 2:
                continue

            token = " ".join(parts[:-1])
            label = parts[-1]

            tokens.append(token)
            labels.append(label)

    # last sentence
    if tokens:
        examples.append({
            "tokens": tokens,
            "labels": labels
        })

    return examples

# 1) read raw examples
examples = read_word_tag_dataset(input_txt_path)

print(f"Total examples: {len(examples)}")
print("First example:")
print(examples[0])

# 2) collect labels automatically
all_labels = sorted({label for ex in examples for label in ex["labels"]})
label2id = {label: i for i, label in enumerate(all_labels)}
id2label = {i: label for label, i in label2id.items()}

print("\nLabels found:")
print(all_labels)
print("\nlabel2id:")
print(label2id)

# 3) convert string labels -> ids
for ex in examples:
    ex["ner_tags"] = [label2id[label] for label in ex["labels"]]
    del ex["labels"]

print("\nFirst example after label->id:")
print(examples[0])

# 4) split 80/20
train_data, val_data = train_test_split(
    examples,
    test_size=0.2,
    random_state=seed,
    shuffle=True
)

print(f"\nTrain size: {len(train_data)}")
print(f"Validation size: {len(val_data)}")

# 5) convert to HuggingFace Dataset
dataset_dict = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(val_data)
})

print(dataset_dict)
print("\nTrain sample:")
print(dataset_dict["train"][0])

Total examples: 900
First example:
{'tokens': ['Him', 'þa', 'Andreas', 'ondsware', 'agef:', '"Hwæt', 'frinest', 'ðu', 'me,', 'frea', 'leofesta,', 'wordum', 'wrætlicum,', 'ond', 'þe', 'wyrda', 'gehwære', 'þurh', 'snyttra', 'cræft', 'soð', 'oncnawest?"'], 'labels': ['O', 'O', 'B-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}

Labels found:
['B-PER', 'O']

label2id:
{'B-PER': 0, 'O': 1}

First example after label->id:
{'tokens': ['Him', 'þa', 'Andreas', 'ondsware', 'agef:', '"Hwæt', 'frinest', 'ðu', 'me,', 'frea', 'leofesta,', 'wordum', 'wrætlicum,', 'ond', 'þe', 'wyrda', 'gehwære', 'þurh', 'snyttra', 'cræft', 'soð', 'oncnawest?"'], 'ner_tags': [1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Train size: 720
Validation size: 180
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 720
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_row

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding=False
    )

    all_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # special tokens like <s>, </s>
            if word_idx is None:
                label_ids.append(-100)

            # first subword of a word
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])

            # remaining subwords of the same word
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

tokenized_datasets = dataset_dict.map(
    tokenize_and_align_labels,
    batched=True
)

print(tokenized_datasets)
print("\nTokenized train sample keys:")
print(tokenized_datasets["train"].column_names)

print("\nExample tokenized sample:")
print(tokenized_datasets["train"][0].keys())
print(tokenized_datasets["train"][0]["input_ids"][:30])
print(tokenized_datasets["train"][0]["labels"][:30])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 720
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 180
    })
})

Tokenized train sample keys:
['tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels']

Example tokenized sample:
dict_keys(['tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'])
[0, 91, 2060, 9772, 4344, 15, 417, 1534, 4450, 15, 417, 38, 1975, 14004, 821, 11819, 741, 1506, 5967, 4636, 6, 18965, 783, 52, 368, 39398, 4636, 6, 2136, 783]
[-100, 1, 0, -100, -100, 1, -100, 0, -100, 1, -100, 0, -100, -100, 1, -100, 1, -100, -100, -100, -100, 1, -100, 1, -100, -100, -100, -100, 1, -100]


In [ ]:
import numpy as np
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

model_checkpoint = "roberta-base"

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

print(model.config)
print("\nModel loaded successfully.")
print(f"Number of labels: {model.config.num_labels}")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "B-PER",
    "1": "O"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "B-PER": 0,
    "O": 1
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}


Model loaded successfully.
Number of labels: 2


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for pred_seq, label_seq in zip(predictions, labels):
        for pred_id, label_id in zip(pred_seq, label_seq):
            # ignore special tokens / non-first-subwords
            if label_id != -100:
                true_predictions.append(pred_id)
                true_labels.append(label_id)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        true_predictions,
        average="micro",
        zero_division=0
    )

    accuracy = accuracy_score(true_labels, true_predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("compute_metrics function is ready.")

compute_metrics function is ready.


In [ ]:
import torch
from transformers import TrainingArguments, Trainer

output_dir = "/content/drive/MyDrive/RoBERTa/finetuned_model"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

training_args = TrainingArguments(
    output_dir=output_dir,

    # evaluation / saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    # training hyperparameters
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,

    # logging
    logging_dir=f"{output_dir}/logs",
    logging_strategy="steps",
    logging_steps=50,

    # efficiency
    fp16=torch.cuda.is_available(),   # H100 မှာ mixed precision
    dataloader_num_workers=4,
    report_to="none",

    # save
    save_total_limit=2,
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


print("Trainer is ready.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


CUDA available: True
GPU: NVIDIA H100 80GB HBM3
Trainer is ready.


In [ ]:
train_result = trainer.train()
print(train_result)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.114037,0.932907,0.932907,0.932907,0.932907
2,No log,0.061452,0.976697,0.976697,0.976697,0.976697
3,0.168299,0.051200,0.981818,0.981818,0.981818,0.981818
4,0.168299,0.050087,0.982330,0.982330,0.982330,0.982330
5,0.036731,0.046921,0.984635,0.984635,0.984635,0.984635


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=115, training_loss=0.0931311856145444, metrics={'train_runtime': 31.1846, 'train_samples_per_second': 115.442, 'train_steps_per_second': 3.688, 'total_flos': 264448648864128.0, 'train_loss': 0.0931311856145444, 'epoch': 5.0})


In [ ]:
eval_result = trainer.evaluate()
print(eval_result)

{'eval_loss': 0.046920787543058395, 'eval_accuracy': 0.9846350832266325, 'eval_precision': 0.9846350832266325, 'eval_recall': 0.9846350832266325, 'eval_f1': 0.9846350832266325, 'eval_runtime': 0.1567, 'eval_samples_per_second': 1148.409, 'eval_steps_per_second': 38.28, 'epoch': 5.0}


In [ ]:
best_model_dir = "/content/drive/MyDrive/RoBERTa/finetuned_model/best_model"

trainer.save_model(best_model_dir)
print(f"Best model saved to: {best_model_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved to: /content/drive/MyDrive/RoBERTa/finetuned_model/best_model


## test on chronicle A


In [ ]:
import pandas as pd

test_path = "/content/D_ChronicleA.xlsx"
test_df = pd.read_excel(test_path)

print("Test size:", len(test_df))
print(test_df.head())
print(test_df.columns.tolist())

Test size: 236
                                            sentence  \
0  þa Cerdic 7 Cynric his sunu cuom up æt Cerdice...   
1  7 þa he gefor, þa feng his sunu Cynric to þam ...   
2  Þa he gefor, þa feng Ceol to þam rice 7 heold ...   
3   Þa feng Cynegils Ceolwulfes broþur sunu to ri...   
4  7 se Cenwalh wæs Cynegilses sunu; 7 þa heold S...   

                      gold_names  
0                 Cerdic, Cynric  
1                         Cynric  
2        Ceol, Ceolwulf, Cerdice  
3  Cynegils, Ceolwulfes, Cenwalh  
4  Cenwalh, Cynegilses, Seaxburg  
['sentence', 'gold_names']


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

best_model_dir = "/content/drive/MyDrive/RoBERTa/finetuned_model/best_model"

device = "cuda" if torch.cuda.is_available() else "cpu"

infer_tokenizer = AutoTokenizer.from_pretrained(best_model_dir, add_prefix_space=True)
infer_model = AutoModelForTokenClassification.from_pretrained(best_model_dir).to(device)
infer_model.eval()

print("Device:", device)
print("Model and tokenizer loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Device: cuda
Model and tokenizer loaded.


In [ ]:
import torch

def predict_names_from_sentence(sentence, model, tokenizer, id2label, device="cuda"):
    words = sentence.split()

    encoding = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    )

    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = model(**encoding)

    predictions = outputs.logits.argmax(dim=-1).squeeze().tolist()
    word_ids = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True
    ).word_ids()

    extracted_names = []
    used_word_idxs = set()

    for pred_id, word_idx in zip(predictions, word_ids):
        if word_idx is None:
            continue

        if word_idx in used_word_idxs:
            continue

        label = id2label[pred_id]

        if label == "B-PER":
            extracted_names.append(words[word_idx])

        used_word_idxs.add(word_idx)

    return extracted_names

In [ ]:
test_df["predicted_names"] = test_df["sentence"].apply(
    lambda x: predict_names_from_sentence(
        sentence=x,
        model=infer_model,
        tokenizer=infer_tokenizer,
        id2label=id2label,
        device=device
    )
)

print(test_df[["sentence", "gold_names", "predicted_names"]].head(10))

                                            sentence  \
0  þa Cerdic 7 Cynric his sunu cuom up æt Cerdice...   
1  7 þa he gefor, þa feng his sunu Cynric to þam ...   
2  Þa he gefor, þa feng Ceol to þam rice 7 heold ...   
3   Þa feng Cynegils Ceolwulfes broþur sunu to ri...   
4  7 se Cenwalh wæs Cynegilses sunu; 7 þa heold S...   
5  Þa feng Æscwine to rice, þæs cyn gęþ to Cerdic...   
6  Þa feng Centwine to Wesseaxna rice Cynegilsing...   
7  Þa feng Ceadwalla to þam rice, þæs cyn gęþ to ...   
8  Þa feng Ęþelheard to, þæs cyn gęþ to Ceardice,...   
9  Þa feng Cuþred to, þæs cyn gęþ to Cerdice, 7 h...   

                      gold_names                     predicted_names  
0                 Cerdic, Cynric              [Cerdic, Cerdicesoran]  
1                         Cynric                                  []  
2        Ceol, Ceolwulf, Cerdice          [Ceol, Ceolwulf, Cerdice.]  
3  Cynegils, Ceolwulfes, Cenwalh     [Cynegils, Ceolwulfes, Cenwalh]  
4  Cenwalh, Cynegilses, Seax

In [ ]:
pred_path = "/content/D_ChronicleA.xlsx"
test_df.to_csv(pred_path, index=False, encoding="utf-8")
print("Saved:", pred_path)

Saved: /content/D_ChronicleA.xlsx


In [ ]:
def parse_gold_names(x):
    if not isinstance(x, str) or x.strip() == "":
        return []

    return [name.strip() for name in x.split(",") if name.strip()]

gold_col = "gold_names"   # make sure correct column

test_df["gold_list"] = test_df[gold_col].apply(parse_gold_names)

print(test_df[[gold_col, "gold_list"]].head(10))

                      gold_names                        gold_list
0                 Cerdic, Cynric                 [Cerdic, Cynric]
1                         Cynric                         [Cynric]
2        Ceol, Ceolwulf, Cerdice        [Ceol, Ceolwulf, Cerdice]
3  Cynegils, Ceolwulfes, Cenwalh  [Cynegils, Ceolwulfes, Cenwalh]
4  Cenwalh, Cynegilses, Seaxburg  [Cenwalh, Cynegilses, Seaxburg]
5                        Æscwine                        [Æscwine]
6                       Centwine                       [Centwine]
7                      Ceadwalla                      [Ceadwalla]
8                      Ęþelheard                      [Ęþelheard]
9                         Cuþred                         [Cuþred]


In [ ]:
def compute_tp_fp_fn(row):
    gold = set(row["gold_list"])
    pred = set(row["predicted_names"])

    tp = len(gold & pred)
    fp = len(pred - gold)
    fn = len(gold - pred)

    return tp, fp, fn

tp_total = 0
fp_total = 0
fn_total = 0

for _, row in test_df.iterrows():
    tp, fp, fn = compute_tp_fp_fn(row)
    tp_total += tp
    fp_total += fp
    fn_total += fn

print("TP:", tp_total)
print("FP:", fp_total)
print("FN:", fn_total)

TP: 302
FP: 205
FN: 151


In [ ]:
precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) > 0 else 0
recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) > 0 else 0

f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("\n===== FINAL RESULT =====")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")


===== FINAL RESULT =====
Precision: 0.5957
Recall:    0.6667
F1 Score:  0.6292
